# NB1 — Data Prep & Phoneme Vocabulary (Iqra'Eval)

**Approach:** phoneme-level mispronunciation detection (wav2vec2-BERT + CTC).

This notebook does NO cutting of words and NO forced alignment. The Iqra'Eval
dataset already ships per-utterance **phoneme transcriptions** (`phoneme_ref`),
so we only need to:
  1. Load `IqraEval/Iqra_train` (real) + optionally `IqraEval/Iqra_TTS` (synthetic, has injected errors).
  2. Build a **phoneme vocabulary** (the CTC label set) from the phoneme strings.
  3. Save the vocab + a tiny manifest. The audio itself is streamed from the HF
     dataset at train time (NB2), so we do not re-save 12 GB of audio here.

Run this on **CPU** (Kaggle or Colab). It is fast — it only reads text columns.

In [ ]:
!pip install -q "datasets==2.21.0" "transformers==4.44.2"

In [ ]:
import json, os
from collections import Counter
from datasets import load_dataset

# ---- PATHS / CONFIG (edit if needed) ------------------------------------
TRAIN_REPO = "IqraEval/Iqra_train"   # columns: id, phoneme_ref, phoneme_aug, audio, sentence, tashkeel_sentence
TTS_REPO   = "IqraEval/Iqra_TTS"     # synthetic; includes injected pronunciation errors
USE_TTS    = True                     # set False to train on real data only first

OUT_DIR    = "/kaggle/working/iqra_prep"   # Colab: "/content/iqra_prep"
os.makedirs(OUT_DIR, exist_ok=True)

# The phoneme column we treat as the GROUND-TRUTH label for what was SAID.
#   - For real data, phoneme_ref = canonical phonemes of the (correct) recitation.
#   - For TTS data, the audio may contain injected errors; we still train the
#     acoustic->phoneme mapping on the phoneme string that matches the audio.
PHONEME_COL = "phoneme_ref"

In [ ]:
# Load only the text columns we need to build the vocab (cheap, no audio decode).
# We stream to avoid pulling 12 GB just to read phoneme strings.
ds_train = load_dataset(TRAIN_REPO, split="train")
print(ds_train)
print("columns:", ds_train.column_names)
ex = ds_train[0]
print("\n--- example row ---")
for k in ds_train.column_names:
    if k != "audio":
        print(f"{k!r}: {ex[k]!r}")
print("audio sr:", ex["audio"]["sampling_rate"], "| n_samples:", len(ex["audio"]["array"]))

In [ ]:
# Phonemes are SPACE-SEPARATED symbols, e.g. "f ii h i nn a ...".
# Shadda/gemination = the consonant symbol DOUBLED (ll, nn, bb, ...);
# long vowels = aa / ii / uu / AA etc. Build the symbol set across the corpus.
#
# IMPORTANT: the TTS audio is MISPRONOUNCED and the matching label is
# `phoneme_mis` (not phoneme_ref). NB2 trains on phoneme_mis for TTS, so we scan
# BOTH phoneme_ref and phoneme_mis here to guarantee the vocab covers every
# symbol that will ever appear as a training target.
def phoneme_tokens(s):
    return [t for t in str(s).split() if t]

counter = Counter()
for s in ds_train[PHONEME_COL]:
    counter.update(phoneme_tokens(s))

if USE_TTS:
    try:
        ds_tts = load_dataset(TTS_REPO, split="train")
        print("TTS columns:", ds_tts.column_names)
        # Scan every phoneme-bearing column so the vocab is a superset.
        for c in ("phoneme_ref", "phoneme_mis", "phoneme_aug"):
            if c in ds_tts.column_names:
                for s in ds_tts[c]:
                    counter.update(phoneme_tokens(s))
                print("  scanned TTS column:", c)
    except Exception as e:
        print("TTS load skipped:", e)

print(f"\nDistinct phoneme symbols: {len(counter)}")
for sym, c in counter.most_common():
    print(f"  {sym:6s} {c}")

In [ ]:
# Build the CTC vocab. Reserve index 0 for CTC blank-equivalent <pad>, and add <unk>.
# (Wav2Vec2CTCTokenizer uses pad_token as the CTC blank.)
symbols = sorted(counter.keys())
vocab = {"<pad>": 0, "<unk>": 1}
for s in symbols:
    vocab[s] = len(vocab)

vocab_path = os.path.join(OUT_DIR, "phoneme_vocab.json")
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

meta = {
    "train_repo": TRAIN_REPO,
    "tts_repo": TTS_REPO if USE_TTS else None,
    "phoneme_col": PHONEME_COL,
    "vocab_size": len(vocab),
    "n_train_rows": len(ds_train),
}
with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("vocab size:", len(vocab))
print("saved:", vocab_path)
print(meta)

In [ ]:
# ---- CACHE THE FULL DATASET TO DISK (the cure for the repeated downloads) ---
# NB1 has already downloaded Iqra_train (and TTS) to build the vocab. Save an
# Arrow copy to /kaggle/working so we NEVER re-download. After this notebook
# finishes (as a COMMIT), click "Create Dataset" on its output, then in NB2 set
# LOCAL_TRAIN_DISK / LOCAL_TTS_DISK to the attached paths.
SAVE_ARROW = True   # set False to only build the vocab and stream in NB2

if SAVE_ARROW:
    # Re-load with all splits (cached already -> fast, no re-download) and save.
    full = load_dataset(TRAIN_REPO)
    full.save_to_disk(os.path.join(OUT_DIR, "iqra_train_arrow"))
    print("saved train+dev Arrow ->", os.path.join(OUT_DIR, "iqra_train_arrow"))

    if USE_TTS:
        try:
            tts_full = load_dataset(TTS_REPO)
            tts_full.save_to_disk(os.path.join(OUT_DIR, "iqra_tts_arrow"))
            print("saved TTS Arrow ->", os.path.join(OUT_DIR, "iqra_tts_arrow"))
        except Exception as e:
            print("TTS save skipped:", e)

    print("DONE. Now: Save Version (Commit) -> 'Create Dataset' on the output -> "
          "attach to NB2 -> set LOCAL_TRAIN_DISK / LOCAL_TTS_DISK.")

### Output of NB1
- `iqra_prep/phoneme_vocab.json` — CTC label set (attach to NB2 & NB3).
- `iqra_prep/meta.json` — record of what was used.
- `iqra_prep/iqra_train_arrow/` + `iqra_prep/iqra_tts_arrow/` — full datasets (Arrow).

**To stop re-downloading every run:** Commit this notebook → "Create Dataset" on
its output → attach to NB2 → set `LOCAL_TRAIN_DISK` / `LOCAL_TTS_DISK`. Then NB2
loads instantly with no network.